# Beijing PM2.5 Forecasting  
## Notebook 04 — Feature Engineering

In this notebook, we create features that help machine learning models
learn temporal patterns in PM2.5 concentration.

Feature engineering is guided by:
- Exploratory Data Analysis (EDA)
- Known physical behavior of air pollution
- Time-series forecasting best practices

The goal is to transform raw measurements into **predictive signals**.

In [35]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Load Cleaned Dataset

We load the cleaned dataset produced in `03_cleaning.ipynb`.
This dataset has:
- controlled missing-value handling
- preserved temporal continuity

In [36]:
DATA_PATH = "../data/processed/aotizhongxin_clean.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"], index_col="datetime")

df.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM
datetime,,,,,,,,,,,,
2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4
2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7
2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6
2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1
2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0


## 2. Why Feature Engineering Is Critical

PM2.5 is not independent across time. Current pollution levels depend heavily on:
- recent past values
- time of day
- season
- meteorological conditions

Raw values alone are insufficient. We therefore introduce **lagged, rolling, and calendar-based features**.


## 3. Define Target Variable

Our prediction target is **PM2.5 at time _t_**. All engineered features must use information from time _t_ or earlier to avoid data leakage.


In [37]:
TARGET = "PM2.5"

## 4. Lag Features (Temporal Memory)

Lag features allow the model to learn:
- persistence
- delayed effects
- short-term trends

We include lags at multiple horizons.

In [38]:
# Create lag features
lag_hours = [1, 3, 6, 12, 24]

for lag in lag_hours:
    df[f"PM2.5_lag_{lag}h"] = df["PM2.5"].shift(lag)

## 5. Rolling Window Statistics

Rolling statistics summarize recent behavior:
- smoothing noise
- capturing short-term trends
- reflecting accumulation effects

In [39]:
# Rolling mean and SD

rolling_windows = [3, 6, 12, 24]

for window in rolling_windows:
    df[f"PM2.5_roll_mean_{window}h"] = (
        df["PM2.5"].rolling(window).mean()
    )
    df[f"PM2.5_roll_std_{window}h"] = (
        df["PM2.5"].rolling(window).std()
    )

## 6. Time-Based Features (Seasonality)

EDA showed strong:
- daily cycles
- seasonal effects

We explicitly encode time information to help the model.

In [40]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

## 7. Cyclical Encoding of Time

Hour of day and month are cyclical:
- hour 23 is close to hour 0
- month 12 is close to month 1

We encode them using sine and cosine transformations.

In [41]:
# Cyclical Features
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

## 8. Wind Direction Encoding

Wind direction is categorical and must be converted to numeric values before regression modeling.

We apply one-hot encoding.

In [42]:
df = pd.get_dummies(
    df,
    columns=["wd"],
    drop_first=True
)

## Other Pollutants as Predictive Features

EDA showed strong correlations between PM2.5 and:
- PM10
- NO₂
- CO
- SO₂
- O₃

We retain these as exogenous predictors.

## 9. Remove Rows Introduced by Lagging

Lag and rolling features introduce missing values at the beginning
of the time series. These rows cannot be used for training.

In [43]:
# Drop NA Rows
df_fe = df.dropna()

print("Rows before feature engineering:", len(df))
print("Rows after feature engineering:", len(df_fe))

Rows before feature engineering: 33902
Rows after feature engineering: 33878


In [45]:
print("Shape after dropping NaNs introduced by lagging:")
print(df_fe.shape)

df_fe.head(10)

Shape after dropping NaNs introduced by lagging:
(33878, 46)


,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM,PM2.5_lag_1h,PM2.5_lag_3h,PM2.5_lag_6h,PM2.5_lag_12h,PM2.5_lag_24h,PM2.5_roll_mean_3h,PM2.5_roll_std_3h,PM2.5_roll_mean_6h,PM2.5_roll_std_6h,PM2.5_roll_mean_12h,PM2.5_roll_std_12h,PM2.5_roll_mean_24h,PM2.5_roll_std_24h,hour,dayofweek,month,hour_sin,hour_cos,month_sin,month_cos,wd_ENE,wd_ESE,wd_N,wd_NE,wd_NNE,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-03-02 00:00:00,22.0,24.0,24.0,44.0,500.0,44.0,-0.4,1031.0,-17.6,0.0,1.4,24.0,12.0,11.0,3.0,4.0,20.333333,4.725816,15.333333,6.377042,11.583333,6.141636,7.875000,5.833282,0,5,3,0.000000,1.000000e+00,1.0,6.123234e-17,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2013-03-02 01:00:00,14.0,17.0,21.0,36.0,400.0,50.0,-1.0,1031.3,-17.3,0.0,1.1,22.0,15.0,8.0,3.0,8.0,20.000000,5.291503,16.333333,5.391351,12.500000,5.535013,8.125000,5.965936,1,5,3,0.258819,9.659258e-01,1.0,6.123234e-17,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2013-03-02 02:00:00,13.0,13.0,20.0,37.0,400.0,47.0,-1.5,1030.9,-16.9,0.0,1.7,14.0,24.0,11.0,6.0,7.0,16.333333,4.932883,16.666667,5.046451,13.083333,5.142662,8.375000,6.041973,2,5,3,0.500000,8.660254e-01,1.0,6.123234e-17,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2013-03-02 03:00:00,3.0,9.0,13.0,34.0,400.0,52.0,-1.4,1030.6,-17.6,0.0,1.4,13.0,22.0,12.0,8.0,6.0,10.000000,6.082763,15.166667,7.467708,12.666667,5.757735,8.250000,6.123724,3,5,3,0.707107,7.071068e-01,1.0,6.123234e-17,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2013-03-02 04:00:00,3.0,7.0,18.0,43.0,400.0,43.0,-1.5,1030.8,-17.7,0.0,0.9,3.0,14.0,15.0,9.0,3.0,6.333333,5.773503,13.166667,8.975894,12.166667,6.336522,8.250000,6.123724,4,5,3,0.866025,5.000000e-01,1.0,6.123234e-17,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2013-03-02 05:00:00,9.0,11.0,19.0,70.0,500.0,20.0,-1.8,1030.1,-17.5,0.0,2.0,3.0,13.0,24.0,10.0,5.0,5.000000,3.464102,10.666667,7.284687,12.083333,6.374072,8.416667,6.085740,5,5,3,0.965926,2.588190e-01,1.0,6.123234e-17,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2013-03-02 06:00:00,4.0,10.0,28.0,46.0,500.0,39.0,-2.5,1029.6,-17.7,0.0,0.7,9.0,3.0,22.0,11.0,3.0,5.333333,3.214550,7.666667,5.046451,11.500000,6.789029,8.458333,6.050362,6,5,3,1.000000,6.123234e-17,1.0,6.123234e-17,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
2013-03-02 07:00:00,3.0,11.0,34.0,58.0,500.0,27.0,-1.7,1029.8,-17.0,0.0,1.2,4.0,3.0,14.0,8.0,3.0,5.333333,3.214550,5.833333,4.215052,11.083333,7.166314,8.458333,6.050362,7,5,3,0.965926,-2.588190e-01,1.0,6.123234e-17,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
2013-03-02 08:00:00,3.0,7.0,21.0,49.0,500.0,43.0,-0.4,1029.6,-17.6,0.0,1.8,3.0,9.0,13.0,11.0,3.0,3.333333,0.577350,4.166667,2.401388,10.416667,7.537281,8.458333,6.050362,8,5,3,0.866025,-5.000000e-01,1.0,6.123234e-17,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False


## Confirmation for feature engineering

In [46]:
# Print Dataset Shape Change
print("Dataset shape after feature creation (before dropna):")
print(df.shape)

Dataset shape after feature creation (before dropna):
(33902, 46)


In [47]:
# Show all feature names
print("Final feature columns:\n")
for col in df.columns:
    print(col)

Final feature columns:

PM2.5
PM10
SO2
NO2
CO
O3
TEMP
PRES
DEWP
RAIN
WSPM
PM2.5_lag_1h
PM2.5_lag_3h
PM2.5_lag_6h
PM2.5_lag_12h
PM2.5_lag_24h
PM2.5_roll_mean_3h
PM2.5_roll_std_3h
PM2.5_roll_mean_6h
PM2.5_roll_std_6h
PM2.5_roll_mean_12h
PM2.5_roll_std_12h
PM2.5_roll_mean_24h
PM2.5_roll_std_24h
hour
dayofweek
month
hour_sin
hour_cos
month_sin
month_cos
wd_ENE
wd_ESE
wd_N
wd_NE
wd_NNE
wd_NNW
wd_NW
wd_S
wd_SE
wd_SSE
wd_SSW
wd_SW
wd_W
wd_WNW
wd_WSW


In [48]:
df.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM,PM2.5_lag_1h,PM2.5_lag_3h,PM2.5_lag_6h,PM2.5_lag_12h,PM2.5_lag_24h,PM2.5_roll_mean_3h,PM2.5_roll_std_3h,PM2.5_roll_mean_6h,PM2.5_roll_std_6h,PM2.5_roll_mean_12h,PM2.5_roll_std_12h,PM2.5_roll_mean_24h,PM2.5_roll_std_24h,hour,dayofweek,month,hour_sin,hour_cos,month_sin,month_cos,wd_ENE,wd_ESE,wd_N,wd_NE,wd_NNE,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,4.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,4,3,0.000000,1.000000,1.0,6.123234e-17,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,4.7,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,4,3,0.258819,0.965926,1.0,6.123234e-17,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,5.6,8.0,NaN,NaN,NaN,NaN,6.333333,2.081666,NaN,NaN,NaN,NaN,NaN,NaN,2,4,3,0.500000,0.866025,1.0,6.123234e-17,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,3.1,7.0,4.0,NaN,NaN,NaN,7.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,3,4,3,0.707107,0.707107,1.0,6.123234e-17,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,2.0,6.0,8.0,NaN,NaN,NaN,5.333333,2.081666,NaN,NaN,NaN,NaN,NaN,NaN,4,4,3,0.866025,0.500000,1.0,6.123234e-17,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False


In [49]:
# To make sure NaN count is zero
df_fe.isna().sum().sort_values(ascending=False)

PM2.5                  0
PM10                   0
SO2                    0
NO2                    0
CO                     0
O3                     0
TEMP                   0
PRES                   0
DEWP                   0
RAIN                   0
WSPM                   0
PM2.5_lag_1h           0
PM2.5_lag_3h           0
PM2.5_lag_6h           0
PM2.5_lag_12h          0
PM2.5_lag_24h          0
PM2.5_roll_mean_3h     0
PM2.5_roll_std_3h      0
PM2.5_roll_mean_6h     0
PM2.5_roll_std_6h      0
PM2.5_roll_mean_12h    0
PM2.5_roll_std_12h     0
PM2.5_roll_mean_24h    0
PM2.5_roll_std_24h     0
hour                   0
dayofweek              0
month                  0
hour_sin               0
hour_cos               0
month_sin              0
month_cos              0
wd_ENE                 0
wd_ESE                 0
wd_N                   0
wd_NE                  0
wd_NNE                 0
wd_NNW                 0
wd_NW                  0
wd_S                   0
wd_SE                  0


In [50]:
df_fe.describe().T

,count,mean,std,min,25%,50%,75%,max
PM2.5,33878.0,82.765391,82.375421,3.000000,22.000000,5.800000e+01,114.000000,898.000000
PM10,33878.0,110.017220,95.765746,2.000000,38.000000,8.600000e+01,154.000000,984.000000
SO2,33878.0,17.218555,22.745263,0.285600,3.000000,9.000000e+00,21.000000,341.000000
NO2,33878.0,59.202511,37.212873,2.000000,30.000000,5.300000e+01,82.000000,290.000000
CO,33878.0,1266.757001,1251.887187,100.000000,500.000000,9.000000e+02,1500.000000,10000.000000
O3,33878.0,55.719550,57.622002,0.214200,7.000000,4.200000e+01,82.000000,423.000000
TEMP,33878.0,13.736268,11.378331,-16.800000,3.400000,1.470000e+01,23.400000,40.500000
PRES,33878.0,1011.764997,10.355314,985.900000,1003.200000,1.011200e+03,1020.000000,1042.000000
DEWP,33878.0,3.278583,13.650855,-35.300000,-7.900000,4.000000e+00,15.700000,28.500000
RAIN,33878.0,0.069299,0.925141,0.000000,0.000000,0.000000e+00,0.000000,72.500000


## Feature Engineering Validation

The following outputs verify that:
- all engineered features were created successfully
- lag and rolling features align correctly
- no missing values remain
- the dataset is ready for modeling

## 8. Save Feature-Engineered Dataset

This dataset will be used for model training and evaluation.

In [51]:
OUTPUT_PATH = "../data/processed/aotizhongxin_features.csv"

df_fe.to_csv(OUTPUT_PATH)

print(f"Feature-engineered dataset saved to: {OUTPUT_PATH}")

Feature-engineered dataset saved to: ../data/processed/aotizhongxin_features.csv


## Feature Set Summary

The final feature set includes:
- lagged PM2.5 values
- rolling statistics
- meteorological variables
- co-pollutant concentrations
- time and seasonal encodings